In [1]:
import xgboost as xgb
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pulp import *
import pandas as pd
import os, glob
import seaborn as sns
from scipy.stats import kruskal
import scikit_posthocs as sp
from scipy.stats import mannwhitneyu
from dotenv import load_dotenv

load_dotenv('./Credentials.env',override=True)

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(os.getenv("GOOGLE_APPLICATION_CREDENTIALS"))  
os.environ['GCLOUD_PROJECT'] =  str(os.getenv("GCLOUD_PROJECT"))
%load_ext google.cloud.bigquery

from google.cloud import bigquery
client=bigquery.Client()
from google.cloud import bigquery_storage_v1

## Read patients_Notes_EHR

In [3]:
%%bigquery Train_set_df
select * from `som-nero-phi-jonc101.blood_culture_stewardship.cohort` where order_year>=2015 and order_year<2022

Query is running:   0%|          |

Downloading:   0%|          |

In [4]:
%%bigquery Val_set_df
select * from `som-nero-phi-jonc101.blood_culture_stewardship.cohort` where order_year>=2022 and order_year<2023

Query is running:   0%|          |

Downloading:   0%|          |

In [5]:
%%bigquery Test_set_df
select * from `som-nero-phi-jonc101.blood_culture_stewardship.cohort` where order_year>=2023 

Query is running:   0%|          |

Downloading:   0%|          |

In [6]:
%%bigquery Notes_df
select * from `som-nero-phi-jonc101.blood_culture_stewardship.EDProviderNotes`

Query is running:   0%|          |

Downloading:   0%|          |

In [7]:
Train_set_df_note=pd.merge(Train_set_df,Notes_df,how='inner',on=['anon_id','pat_enc_csn_id_coded','order_proc_id_coded'])
Train_set_df_note[['anon_id','pat_enc_csn_id_coded','order_proc_id_coded']].drop_duplicates().shape

Val_set_df_note=pd.merge(Val_set_df,Notes_df,how='inner',on=['anon_id','pat_enc_csn_id_coded','order_proc_id_coded'])
Val_set_df_note[['anon_id','pat_enc_csn_id_coded','order_proc_id_coded']].drop_duplicates().shape

Test_set_df_note=pd.merge(Test_set_df,Notes_df,how='inner',on=['anon_id','pat_enc_csn_id_coded','order_proc_id_coded'])
Test_set_df_note[['anon_id','pat_enc_csn_id_coded','order_proc_id_coded']].drop_duplicates().shape

(25138, 3)

In [8]:
Train_set_df_note=Train_set_df_note[Train_set_df_note.deid_note_text.notnull()]
Train_set_df_note[['anon_id','pat_enc_csn_id_coded','order_proc_id_coded']].drop_duplicates().shape

Val_set_df_note=Val_set_df_note[Val_set_df_note.deid_note_text.notnull()]
Val_set_df_note[['anon_id','pat_enc_csn_id_coded','order_proc_id_coded']].drop_duplicates().shape

Test_set_df_note=Test_set_df_note[Test_set_df_note.deid_note_text.notnull()]
Test_set_df_note[['anon_id','pat_enc_csn_id_coded','order_proc_id_coded']].drop_duplicates().shape

(25138, 3)

In [9]:
Train_set_df_note.drop_duplicates(subset=['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded','blood_culture_order_datetime','notedatetime','deid_note_text'],inplace=True)
Train_set_df_note.shape

Val_set_df_note.drop_duplicates(subset=['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded','blood_culture_order_datetime','notedatetime','deid_note_text'],inplace=True)
Val_set_df_note.shape

Test_set_df_note.drop_duplicates(subset=['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded','blood_culture_order_datetime','notedatetime','deid_note_text'],inplace=True)
Test_set_df_note.shape


(26247, 113)

In [10]:
Train_set_df_note['blood_culture_order_datetime'] = pd.to_datetime(Train_set_df_note['blood_culture_order_datetime'])
Train_set_df_note['notedatetime'] = pd.to_datetime(Train_set_df_note['notedatetime'])
Train_set_df_note['time-diff']=(Train_set_df_note['blood_culture_order_datetime']-Train_set_df_note['notedatetime']).dt.total_seconds()/3600
Train_set_df_note['time-diff']=Train_set_df_note['time-diff'].astype(int)
Train_set_df_note=Train_set_df_note[Train_set_df_note['time-diff']>=-24]


Val_set_df_note['blood_culture_order_datetime'] = pd.to_datetime(Val_set_df_note['blood_culture_order_datetime'])
Val_set_df_note['notedatetime'] = pd.to_datetime(Val_set_df_note['notedatetime'])
Val_set_df_note['time-diff']=(Val_set_df_note['blood_culture_order_datetime']-Val_set_df_note['notedatetime']).dt.total_seconds()/3600
Val_set_df_note['time-diff']=Val_set_df_note['time-diff'].astype(int)
Val_set_df_note=Val_set_df_note[Val_set_df_note['time-diff']>=-24]

Test_set_df_note['blood_culture_order_datetime'] = pd.to_datetime(Test_set_df_note['blood_culture_order_datetime'])
Test_set_df_note['notedatetime'] = pd.to_datetime(Test_set_df_note['notedatetime'])
Test_set_df_note['time-diff']=(Test_set_df_note['blood_culture_order_datetime']-Test_set_df_note['notedatetime']).dt.total_seconds()/3600
Test_set_df_note['time-diff']=Test_set_df_note['time-diff'].astype(int)
Test_set_df_note=Test_set_df_note[Test_set_df_note['time-diff']>=-24]


In [11]:
Train_set_df_note['time-diff']=(Train_set_df_note['blood_culture_order_datetime']-Train_set_df_note['notedatetime']).dt.total_seconds().abs()
Val_set_df_note['time-diff']=(Val_set_df_note['blood_culture_order_datetime']-Val_set_df_note['notedatetime']).dt.total_seconds().abs()
Test_set_df_note['time-diff']=(Test_set_df_note['blood_culture_order_datetime']-Test_set_df_note['notedatetime']).dt.total_seconds().abs()


Train_set_df_note = Train_set_df_note.sort_values('time-diff').drop_duplicates(subset=['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'blood_culture_order_datetime'], keep='first')
Val_set_df_note = Val_set_df_note.sort_values('time-diff').drop_duplicates(subset=['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'blood_culture_order_datetime'], keep='first')
Test_set_df_note = Test_set_df_note.sort_values('time-diff').drop_duplicates(subset=['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'blood_culture_order_datetime'], keep='first')


In [12]:
Train_set_df_note['doc_id'] = Train_set_df_note.groupby(['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded','blood_culture_order_datetime']).ngroup() + 1
Val_set_df_note['doc_id'] = Val_set_df_note.groupby(['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded','blood_culture_order_datetime']).ngroup() + 1
Test_set_df_note['doc_id'] = Test_set_df_note.groupby(['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded','blood_culture_order_datetime']).ngroup() + 1

## Evalute GPT4 on Fabre : whole Notes + EHR 

In [13]:
Test_set_df_note['Label']=(Test_set_df_note['positive_blood_culture']| Test_set_df_note['positive_blood_culture_in_week'])

In [14]:
EHR=[col for col in Test_set_df_note.columns.values if 'min' in col]
EHR.extend([col for col in Test_set_df_note.columns.values if 'max' in col])
EHR.extend([col for col in Test_set_df_note.columns.values if 'avg' in col])
EHR.extend([col for col in Test_set_df_note.columns.values if 'median' in col])
EHR

['min_heartrate',
 'min_resprate',
 'min_temp',
 'min_sysbp',
 'min_diasbp',
 'min_wbc',
 'min_neutrophils',
 'min_lymphocytes',
 'min_hgb',
 'min_plt',
 'min_na',
 'min_hco3',
 'min_bun',
 'min_cr',
 'min_lactate',
 'min_procalcitonin',
 'max_heartrate',
 'max_resprate',
 'max_temp',
 'max_sysbp',
 'max_diasbp',
 'max_wbc',
 'max_neutrophils',
 'max_lymphocytes',
 'max_hgb',
 'max_plt',
 'max_na',
 'max_hco3',
 'max_bun',
 'max_cr',
 'max_lactate',
 'max_procalcitonin',
 'avg_heartrate',
 'avg_resprate',
 'avg_temp',
 'avg_sysbp',
 'avg_diasbp',
 'avg_wbc',
 'avg_neutrophils',
 'avg_lymphocytes',
 'avg_hgb',
 'avg_plt',
 'avg_na',
 'avg_hco3',
 'avg_bun',
 'avg_cr',
 'avg_lactate',
 'avg_procalcitonin',
 'median_heartrate',
 'median_resprate',
 'median_temp',
 'median_sysbp',
 'median_diasbp',
 'median_wbc',
 'median_neutrophils',
 'median_lymphocytes',
 'median_hgb',
 'median_plt',
 'median_na',
 'median_hco3',
 'median_bun',
 'median_cr',
 'median_lactate',
 'median_procalcitonin']

In [15]:
def format_structured_features(row, ehr_columns):
    return "\n".join([f"- {col}: {row[col]}" for col in ehr_columns if pd.notnull(row[col])])

prompt_records = []

def split_note_text(note_text, max_words=1000, overlap=10):
    words = note_text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = words[i:i+max_words]
        chunks.append(" ".join(chunk))
        i += max_words - overlap
    return chunks

for i, row in Test_set_df_note.iterrows():
    structured_block = format_structured_features(row, EHR)
    note_text = row['deid_note_text']
    note_text = re.sub(r'\s+', ' ', note_text.replace('\n', ' ')).strip()
    
    # Split if longer than 3000 words
    if len(note_text.split()) > 1000:
        note_chunks = split_note_text(note_text)
    else:
        note_chunks = [note_text]

    # Generate one prompt per chunk
    for chunk_index, chunk in enumerate(note_chunks):
        prompt = (
                "You are a physician. From the following patinet information, identify which of the following conditions are present or can be inferred. \n"
                "Condition List:CLABSI, CRBSI, Discitis, Native vertebral osteomyelitis, \n"
                "Epidural abscess, Meningitis, Nontraumatic native septic arthritis, \n"
                "Ventriculoarterial shunt infection, Severe sepsis, Septic shock, \n"
                "Infective endocarditis, Endovascular infection, Septic thrombophlebitis,\n" 
                "Infected endovascular thrombi, ICD lead infection, Pacemaker lead infection, Intravascular catheter infection,\n"
                "Vascular graft infection, Acute pyelonephritis, Cholangitis, Nonvascular shunt infection, \n"
                "Prosthetic vertebral osteomyelitis, Severe community-acquired pneumonia (CAP) PSI IV-V,\n"
                "Cellulitis with comorbidities, Ventilator-associated pneumonia (VAP), Isolated fever, Leukocytosis,\n"
                "Nonsevere cellulitis, Lower UTI, Cystitis, Prostatitis, Postoperative fever within 48h of surgery. \n"
                 "Patient Information:\n"
                 f"{structured_block}\n\n"
                 f"Clinical Note:: {chunk}.\n"
                "Respond with *only* a comma-separated list of symptoms that are present."
        )
        prompt_records.append({
            'anon_id': row['anon_id'],
            'pat_enc_csn_id_coded': row['pat_enc_csn_id_coded'],
            'order_proc_id_coded': row['order_proc_id_coded'],
            'blood_culture_order_datetime':row['blood_culture_order_datetime'],
            'Label':row['Label'],
            'doc_id':row['doc_id'],
            'chunk_index': chunk_index,
            'prompt': prompt
        })

# Convert to DataFrame
Test_set_prompt_df = pd.DataFrame(prompt_records)

In [16]:
Test_set_prompt_df=Test_set_prompt_df.sort_values(by=['doc_id'])Test_set_prompt_df.to_csv('Fabre_Testset_WholeNotes_EHR_prompts.csv',index=False)

In [20]:
import requests
import json
my_key ='5f57674c921c459aa144c7e35360a735'
headers = {'Ocp-Apim-Subscription-Key': my_key, 'Content-Type': 'application/json'}

url = "https://apim.stanfordhealthcare.org/openai-eastus2/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview" 


In [21]:
import pdb
import json
import requests
import pdb
from time import sleep

def gpt4response(row, model="gpt-4"):
    
    prompt=row['prompt']
    doc_id=row['doc_id']
    if not prompt:
        return None

    payload = json.dumps({
        "model": model,
        "messages": [{"role": "user", "content": prompt}]
    })

    while True:
        try:
            response = requests.post(url, headers=headers, data=payload)
            response_json = response.json()
            if 'choices' in response_json:
                return response_json['choices'][0]['message']['content']
            else:
                print(f"doc_id:{doc_id}")
                print("Response missing 'choices'. Retrying in 10 seconds...")
                sleep(10)
        except Exception as e:
            print(f"Error: {e}. Retrying in 10 seconds...")
            sleep(10)

In [23]:
len_batches=100
n_batches=int(Test_set_prompt_df.shape[0]/len_batches)
for i in range(546,n_batches):
    DF=Test_set_prompt_df.iloc[i*len_batches:i*len_batches+len_batches]
    DF['GPT_response']=DF.apply(lambda row:gpt4response(row),axis=1)
    filename=f"GPT4_Testset_Fabre_WholeNotes_EHR_part{i}.csv"
    DF.to_csv(filename,index=False)
    print(f'Batch{i} is done')
print(i)

doc_id:24621
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24621
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24622
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24623
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24627
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24627
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24627
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24630
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24633
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24633
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24633
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24638
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24642
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24642
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24642
Response missing 'choices'. Retrying in 10 second

/var/folders/3b/31jnsf8150z57g6vqxxwj3vw0000gp/T/ipykernel_18101/110673075.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  DF['GPT_response']=DF.apply(lambda row:gpt4response(row),axis=1)


Batch546 is done
doc_id:24655
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24655
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24655
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24657
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24657
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24662
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24662
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24662
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24663
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24665
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24665
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24665
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24667
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24669
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24669
Response missing 'choices'. Retr

/var/folders/3b/31jnsf8150z57g6vqxxwj3vw0000gp/T/ipykernel_18101/110673075.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  DF['GPT_response']=DF.apply(lambda row:gpt4response(row),axis=1)


Batch547 is done
doc_id:24706
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24707
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24709
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24709
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24710
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24711
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24712
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24715
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24715
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24716
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24717
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24718
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24722
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24722
Response missing 'choices'. Retrying in 10 seconds...
doc_id:24723
Response missing 'choices'. Retr

...

/var/folders/3b/31jnsf8150z57g6vqxxwj3vw0000gp/T/ipykernel_18101/110673075.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  DF['GPT_response']=DF.apply(lambda row:gpt4response(row),axis=1)


Batch555 is done
doc_id:25050
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25054
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25054
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25054
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25058
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25058
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25059
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25059
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25059
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25061
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25061
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25061
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25061
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25063
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25064
Response missing 'choices'. Retr

/var/folders/3b/31jnsf8150z57g6vqxxwj3vw0000gp/T/ipykernel_18101/110673075.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  DF['GPT_response']=DF.apply(lambda row:gpt4response(row),axis=1)


Batch556 is done
doc_id:25087
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25088
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25089
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25091
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25092
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25094
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25095
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25096
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25100
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25101
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25101
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25103
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25108
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25109
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25110
Response missing 'choices'. Retr

/var/folders/3b/31jnsf8150z57g6vqxxwj3vw0000gp/T/ipykernel_18101/110673075.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  DF['GPT_response']=DF.apply(lambda row:gpt4response(row),axis=1)


In [24]:
DF=Test_set_prompt_df.iloc[i*len_batches:]
DF['GPT_response']=DF.apply(lambda row:gpt4response(row),axis=1)
filename=f"GPT4_Testset_Fabre_WholeNotes_EHR_part{i+1}.csv"
DF.to_csv(filename,index=False)
print(f'Batch{i} is done')

doc_id:25125
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25126
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25127
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25131
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25132
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25132
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25133
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25133
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25136
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25137
Response missing 'choices'. Retrying in 10 seconds...
doc_id:25137
Response missing 'choices'. Retrying in 10 seconds...
Batch558 is done


/var/folders/3b/31jnsf8150z57g6vqxxwj3vw0000gp/T/ipykernel_18101/2343277316.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  DF['GPT_response']=DF.apply(lambda row:gpt4response(row),axis=1)


### Evalute_GPT4 responses

In [ ]:
import glob
Res=[]
i=0
for file in glob.glob('./*.csv'):
    if 'GPT4_Testset_Fabre_WholeNotes_EHR_part' in file:
        i+=1
        if not len(Res):
            Res=pd.read_csv(file)
        else:
            Res=pd.concat([Res,pd.read_csv(file)],axis=0)
Res.head()

In [28]:
high_risk=['CLABSI', 'CRBSI', 'Discitis', 'Native vertebral osteomyelitis',
    'Epidural abscess', 'Meningitis', 'Nontraumatic native septic arthritis',
    'Ventriculoarterial shunt infection', 'Severe sepsis', 'Septic shock',
    'Infective endocarditis', 'Endovascular infection',
    'Septic thrombophlebitis', 'Infected endovascular thrombi','Sepsis',
    'Implantable cardioverter defibrillator (ICD) infections',
    'ICD lead infection',
    'Pacemaker lead infection', 'Intravascular catheter infection',
    'Vascular graft infection']
intermediate_risk=[
    'Acute pyelonephritis',
    'Cholangitis',
    'Nonvascular shunt infection',
    'Prosthetic vertebral osteomyelitis',
    'Severe community acquired pneumonia',
    'CAP',
    'PSI V',
    'IV',
    'Cellulitis with comorbidities',
    'Ventilator associated pneumonia',
    'VAP'
]
low_risk=[
    'Isolated fever',
    'leukocytosis',
    'Nonsevere cellulitis',
    'Lower UTI',
    'cystitis',
    'prostatitis',
    'Postop fever within 48 hours of surgery'
]

In [29]:
def Apply_Fabre(x):
    for concept in high_risk:
        if concept.lower() in x.lower():
            return 1
    for concept in intermediate_risk:
        if concept.lower() in x.lower():
            return 1
    for concept in low_risk:
        if concept.lower() in x.lower():
            return 0
    print(x)
    return -1         

In [30]:
Res['GPT_response']=Res['GPT_response'].astype(str)
Res['Gpt_Label_Fabre']=Res['GPT_response'].apply(lambda x:Apply_Fabre(x))

altered mental status, fall, elevated troponin, history of dementia, diffuse extremity pain, leukocyturia, moderate bacteriuria on urinalysis, tenderness present
fall, altered mental status, pain all over, mild midline neck tenderness, cervical back tenderness
altered mental status,fall,diffuse extremity pain,tenderness,back tenderness,warm and dry skin,oriented to person,history of dementia,elevated troponin,weakness,leukocyturia,bacteriuria in urine,history of fibromyalgia
nan
Chills, fever, cough, hypoxemia (low oxygen saturation at home)
COVID, Hypoxemia, Fever, Hypotension, Elevated BNP, Elevated creatinine, Elevated BUN, Mild anemia, Lymphopenia, Troponin elevation
COVID, Hypoxemia, Hypotension, Fever, Elevated BNP, Mildly elevated troponin, Anemia (Hb 11.4), Mildly elevated creatinine, Elevated BUN, Mild leukopenia (Low absolute lymphocytes), Ventricular paced rhythm
chills,fever,cough,low oxygen saturation
Fever,Headache,Myalgias,Fatigue,Sinus pain
headache, fever, neck stiffne

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [31]:
Res=Res[['anon_id','pat_enc_csn_id_coded','order_proc_id_coded','blood_culture_order_datetime','Label','Gpt_Label_Fabre']]
Res.drop_duplicates(inplace=True)
test=Res.groupby(by=['anon_id','pat_enc_csn_id_coded','order_proc_id_coded','blood_culture_order_datetime','Label'],as_index=False).max()
test.drop_duplicates(inplace=True)
test
test['Label']=test['Label'].astype('int')

In [32]:
test=test[test.Gpt_Label_Fabre.isin([0,1])]
print(test['Label'].unique())
print(test['Gpt_Label_Fabre'].unique())
print(test.isnull().sum())

[1 0]
[1 0]
anon_id                         0
pat_enc_csn_id_coded            0
order_proc_id_coded             0
blood_culture_order_datetime    0
Label                           0
Gpt_Label_Fabre                 0
dtype: int64


In [33]:
from sklearn.metrics import confusion_matrix, f1_score
import pandas as pd

test['Label'] = test['Label'].astype(int)
test['Gpt_Label_Fabre'] = test['Gpt_Label_Fabre'].astype(int)

y_true = test['Label']
y_pred =test['Gpt_Label_Fabre']

# Confusion matrix: [[TN, FP], [FN, TP]]
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

# Metrics
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
ppv         = tp / (tp + fp) if (tp + fp) > 0 else 0
npv         = tn / (tn + fn) if (tn + fn) > 0 else 0
f1          = f1_score(y_true, y_pred)

# Print results
print(f"Sensitivity (Recall): {sensitivity:.2f}")
print(f"Specificity: {specificity:.2f}")
print(f"PPV (Precision): {ppv:.2f}")
print(f"NPV: {npv:.2f}")
print(f"F1 Score: {f1:.2f}")

Sensitivity (Recall): 0.81
Specificity: 0.29
PPV (Precision): 0.07
NPV: 0.96
F1 Score: 0.12
